In [1]:
# Project path setup: works whether Jupyter was launched from the project root or notebooks/.
from pathlib import Path
import os
cwd = Path.cwd()
if (cwd / "data").is_dir() and (cwd / "notebooks").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").is_dir() and cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Open this notebook from the project root or its notebooks/ folder.")
os.chdir(PROJECT_ROOT / "notebooks")
print("Project root:", PROJECT_ROOT)
print("Notebook working directory:", Path.cwd())

Project root: /home/akshat/Desktop/main files (i think)/ml course work/ml project
Notebook working directory: /home/akshat/Desktop/main files (i think)/ml course work/ml project/notebooks


# 06 — Controlled Model Improvement and Hyperparameter Tuning

This notebook compares stop-word policies, regularisation, LinearSVC settings, tree models, and handcrafted review-style features. The best raw CV candidate is recorded here; it is not automatically the final website model.

**Primary metric:** cross-validation Macro F1. Training Macro F1 and the train–validation gap are diagnostics. The held-out test set is not used for the tuning searches.

The final deployment decision is made after considering predictive performance, generalisation gap, model size, inference requirements, and the need for probability output. The project ultimately retains Logistic Regression for deployment; see Notebook 10.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
import joblib
import re
from pathlib import Path

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate
from sklearn.metrics import f1_score
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, FunctionTransformer

from xgboost import XGBClassifier

RANDOM_STATE = 42
N_JOBS = 3

In [3]:
train_df = pd.read_csv("../data/processed/train.csv")

assert train_df["text"].notna().all(), "Missing review texts found."
assert train_df["target"].notna().all(), "Missing target labels found."

X_train = train_df["text"].astype(str)
y_train = train_df["target"].astype(int)

print("Training samples:", len(X_train))
print("Class distribution:")
print(y_train.value_counts().sort_index())

Training samples: 32328
Class distribution:
target
0    16172
1    16156
Name: count, dtype: int64


In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc"
}

OUTPUT_DIR = Path("../results/improvement_no_stopwords")
MODEL_DIR = Path("../models/improvement_no_stopwords")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Results:", OUTPUT_DIR.resolve())
print("Models:", MODEL_DIR.resolve())

Results: /home/akshat/Desktop/main files (i think)/ml course work/ml project/results/improvement_no_stopwords
Models: /home/akshat/Desktop/main files (i think)/ml course work/ml project/models/improvement_no_stopwords


## Experiment A — Stop-word comparison

This is a controlled comparison. It holds each model's current classifier settings and TF-IDF settings fixed, changing only the stop-word policy. The comparison is not a search for the best overall configuration yet.

Do not assume that retaining all words is better: `None` retains potentially useful negations but also retains common/noisy terms. The CV results decide.

In [5]:
fixed_linear_models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2), min_df=3, max_df=1.0,
            sublinear_tf=True, max_features=None
        )),
        ("classifier", LogisticRegression(
            C=3, class_weight="balanced",
            solver="liblinear", max_iter=3000, random_state=RANDOM_STATE
        ))
    ]),
    "LinearSVC": Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2), min_df=3, max_df=0.95,
            sublinear_tf=True, max_features=None
        )),
        ("classifier", LinearSVC(
            C=0.1, class_weight="balanced",
            max_iter=5000, random_state=RANDOM_STATE
        ))
    ])
}

stopword_rows = []

for model_name, base_pipeline in fixed_linear_models.items():
    for stop_words in [None, "english"]:
        pipeline = clone(base_pipeline)
        pipeline.set_params(tfidf__stop_words=stop_words)

        scores = cross_validate(
            pipeline, X_train, y_train,
            cv=cv, scoring=scoring,
            return_train_score=True,
            n_jobs=N_JOBS, pre_dispatch=N_JOBS
        )

        stopword_rows.append({
            "Model": model_name,
            "Stop Words": "None" if stop_words is None else "English",
            "Training Macro F1": scores["train_macro_f1"].mean(),
            "CV Macro F1": scores["test_macro_f1"].mean(),
            "Generalization Gap": (
                scores["train_macro_f1"].mean()
                - scores["test_macro_f1"].mean()
            ),
            "CV Macro F1 Std": scores["test_macro_f1"].std(),
            "CV Accuracy": scores["test_accuracy"].mean(),
            "CV Balanced Accuracy": scores["test_balanced_accuracy"].mean(),
            "CV ROC-AUC": scores["test_roc_auc"].mean()
        })

stopword_comparison = pd.DataFrame(stopword_rows).sort_values(
    "CV Macro F1", ascending=False
)
display(stopword_comparison.round(4))
stopword_comparison.to_csv(OUTPUT_DIR / "A_stopword_comparison.csv", index=False)

,Model,Stop Words,Training Macro F1,CV Macro F1,Generalization Gap,CV Macro F1 Std,CV Accuracy,CV Balanced Accuracy,CV ROC-AUC
0,Logistic Regression,None,0.9839,0.9385,0.0454,0.0023,0.9385,0.9385,0.9847
2,LinearSVC,None,0.9650,0.9330,0.0320,0.0019,0.9330,0.9330,0.9819
1,Logistic Regression,English,0.9709,0.8959,0.0750,0.0032,0.8959,0.8959,0.9628
3,LinearSVC,English,0.9400,0.8884,0.0516,0.0032,0.8884,0.8884,0.9571


**Interpretation:** choose the stop-word setting using CV Macro F1, while checking fold variability and the train–validation gap. If the scores are close, prefer the simpler/more stable choice and discuss the trade-off. Do not select based on training F1.

## Experiment B — Linear models: Ridge, Lasso, Elastic Net and C

Logistic Regression currently uses L2 regularisation by default. This experiment explicitly compares:

- `l2`: Ridge-style regularisation
- `l1`: Lasso-style regularisation
- `elasticnet`: a mixture of L1 and L2

The `C` parameter is inverse regularisation strength: smaller `C` means stronger regularisation. Elastic Net uses `saga`.

In [6]:
# Set this after reviewing Experiment A.
# Allowed values: None or "english".
# Default here is English because the concern is noisy/common tokens;
# change to None only if Experiment A supports it.
SELECTED_STOP_WORDS = None

linear_tfidf_parameters = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [3, 5, 10],
    "tfidf__max_df": [0.95, 1.0],
    "tfidf__sublinear_tf": [True, False],
    "tfidf__max_features": [None, 50000, 100000],
    "tfidf__stop_words": [SELECTED_STOP_WORDS]
}

logistic_parameter_distributions = [
    {
        **linear_tfidf_parameters,
        "classifier__penalty": ["l1", "l2"],
        "classifier__solver": ["liblinear"],
        "classifier__C": [0.03, 0.1, 0.3, 1, 3],
        "classifier__class_weight": [None, "balanced"]
    },
    {
        **linear_tfidf_parameters,
        "classifier__penalty": ["elasticnet"],
        "classifier__solver": ["saga"],
        "classifier__l1_ratio": [0.25, 0.5, 0.75],
        "classifier__C": [0.03, 0.1, 0.3, 1, 3],
        "classifier__class_weight": [None, "balanced"]
    }
]

In [7]:
def run_search(
    model_name, estimator, param_distributions, n_iter, output_prefix
):
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring=scoring,
        refit="macro_f1",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        pre_dispatch=N_JOBS,
        verbose=2,
        return_train_score=True,
        error_score="raise"
    )

    start = time.time()
    search.fit(X_train, y_train)
    elapsed = time.time() - start

    results = pd.DataFrame(search.cv_results_)
    results["generalization_gap"] = (
        results["mean_train_macro_f1"] - results["mean_test_macro_f1"]
    )
    results.to_csv(OUTPUT_DIR / f"{output_prefix}_search_results.csv", index=False)
    joblib.dump(
        search.best_estimator_,
        MODEL_DIR / f"{output_prefix}_best_pipeline.joblib"
    )

    print(f"\n{model_name} completed in {elapsed/60:.2f} minutes")
    print("Best CV Macro F1:", round(search.best_score_, 5))
    print("Best parameters:", search.best_params_)
    print("Training Macro F1:", round(
        search.cv_results_["mean_train_macro_f1"][search.best_index_], 5
    ))
    print("Generalisation gap:", round(
        search.cv_results_["mean_train_macro_f1"][search.best_index_]
        - search.best_score_, 5
    ))

    return search, results

logistic_search, logistic_results = run_search(
    "Logistic Regression (L1/L2/Elastic Net)",
    Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("classifier", LogisticRegression(
            max_iter=5000, random_state=RANDOM_STATE
        ))
    ]),
    logistic_parameter_distributions,
    n_iter=30,
    output_prefix="B_logistic_regularisation"
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END classifier__C=3, classifier__class_weight=None, classifier__l1_ratio=0.25, classifier__penalty=elasticnet, classifier__solver=saga, tfidf__max_df=0.95, tfidf__max_features=None, tfidf__min_df=5, tfidf__ngram_range=(1, 2), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=  35.9s
[CV] END classifier__C=3, classifier__class_weight=None, classifier__l1_ratio=0.25, classifier__penalty=elasticnet, classifier__solver=saga, tfidf__max_df=0.95, tfidf__max_features=None, tfidf__min_df=5, tfidf__ngram_range=(1, 2), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=  36.0s
[CV] END classifier__C=3, classifier__class_weight=None, classifier__l1_ratio=0.25, classifier__penalty=elasticnet, classifier__solver=saga, tfidf__max_df=0.95, tfidf__max_features=None, tfidf__min_df=5, tfidf__ngram_range=(1, 2), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=  36.9s
[CV] END classifier__C=3, classifier__

## Experiment C — LinearSVC regularisation

LinearSVC uses L2 regularisation by default. We tune `C` while also allowing the TF-IDF vocabulary and n-gram representation to vary. This tests whether stronger regularisation improves validation performance without forcing the train–validation gap to zero.

In [8]:
svm_parameter_distributions = {
    **linear_tfidf_parameters,
    "classifier__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1],
    "classifier__class_weight": [None, "balanced"]
}

svm_search, svm_results = run_search(
    "LinearSVC",
    Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("classifier", LinearSVC(
            max_iter=10000, random_state=RANDOM_STATE
        ))
    ]),
    svm_parameter_distributions,
    n_iter=24,
    output_prefix="C_linearsvc"
)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
[CV] END classifier__C=0.01, classifier__class_weight=None, tfidf__max_df=1.0, tfidf__max_features=50000, tfidf__min_df=5, tfidf__ngram_range=(1, 2), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=   2.8s
[CV] END classifier__C=0.01, classifier__class_weight=None, tfidf__max_df=1.0, tfidf__max_features=50000, tfidf__min_df=5, tfidf__ngram_range=(1, 2), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=   2.8s
[CV] END classifier__C=0.01, classifier__class_weight=None, tfidf__max_df=1.0, tfidf__max_features=50000, tfidf__min_df=5, tfidf__ngram_range=(1, 2), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=   2.8s
[CV] END classifier__C=1, classifier__class_weight=None, tfidf__max_df=0.95, tfidf__max_features=None, tfidf__min_df=3, tfidf__ngram_range=(1, 1), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=   0.9s
[CV] END classifier__C=1, classifier__class_weight=None, tfidf__m

## Experiment D — Tree models: larger vocabulary and more expressive trees

The earlier tree search used only 5,000–10,000 TF-IDF features and restrictive depth/leaf settings. Here we expand the vocabulary and permit more expressive trees.

This is a new search space, not a perfectly isolated causal test: several settings are searched together. The best results should be interpreted as evidence about the expanded search, not proof that vocabulary size alone caused an improvement.

In [9]:
tree_tfidf_parameters = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [3, 5, 10],
    "tfidf__max_df": [0.95, 1.0],
    "tfidf__sublinear_tf": [True, False],
    "tfidf__max_features": [20000, 40000, 60000],
    "tfidf__stop_words": [SELECTED_STOP_WORDS]
}

rf_parameters = {
    **tree_tfidf_parameters,
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [20, 30, None],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2, 5],
    "classifier__max_features": ["sqrt", 0.01],
    "classifier__class_weight": [None, "balanced"]
}

rf_search, rf_results = run_search(
    "Random Forest",
    Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("classifier", RandomForestClassifier(
            random_state=RANDOM_STATE, n_jobs=1
        ))
    ]),
    rf_parameters,
    n_iter=12,
    output_prefix="D_random_forest"
)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END classifier__class_weight=balanced, classifier__max_depth=30, classifier__max_features=sqrt, classifier__min_samples_leaf=2, classifier__min_samples_split=2, classifier__n_estimators=100, tfidf__max_df=1.0, tfidf__max_features=60000, tfidf__min_df=10, tfidf__ngram_range=(1, 2), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=  10.0s
[CV] END classifier__class_weight=balanced, classifier__max_depth=30, classifier__max_features=sqrt, classifier__min_samples_leaf=2, classifier__min_samples_split=2, classifier__n_estimators=100, tfidf__max_df=1.0, tfidf__max_features=60000, tfidf__min_df=10, tfidf__ngram_range=(1, 2), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=  10.2s
[CV] END classifier__class_weight=balanced, classifier__max_depth=30, classifier__max_features=sqrt, classifier__min_samples_leaf=2, classifier__min_samples_split=2, classifier__n_estimators=100, tfidf__max_df=1.0, tfidf__max_f

In [10]:
xgb_parameters = {
    **tree_tfidf_parameters,
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [3, 4, 6],
    "classifier__learning_rate": [0.03, 0.05, 0.1],
    "classifier__min_child_weight": [1, 3, 5],
    "classifier__subsample": [0.7, 0.85, 1.0],
    "classifier__colsample_bytree": [0.5, 0.7, 0.9],
    "classifier__reg_lambda": [1, 5, 10],
    "classifier__reg_alpha": [0, 0.1, 1],
    "classifier__gamma": [0, 0.1, 0.5]
}

xgb_search, xgb_results = run_search(
    "XGBoost",
    Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("classifier", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=1
        ))
    ]),
    xgb_parameters,
    n_iter=15,
    output_prefix="D_xgboost"
)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
[CV] END classifier__colsample_bytree=0.5, classifier__gamma=0, classifier__learning_rate=0.1, classifier__max_depth=3, classifier__min_child_weight=5, classifier__n_estimators=300, classifier__reg_alpha=1, classifier__reg_lambda=1, classifier__subsample=0.85, tfidf__max_df=1.0, tfidf__max_features=60000, tfidf__min_df=3, tfidf__ngram_range=(1, 2), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=  29.8s
[CV] END classifier__colsample_bytree=0.5, classifier__gamma=0, classifier__learning_rate=0.1, classifier__max_depth=3, classifier__min_child_weight=5, classifier__n_estimators=300, classifier__reg_alpha=1, classifier__reg_lambda=1, classifier__subsample=0.85, tfidf__max_df=1.0, tfidf__max_features=60000, tfidf__min_df=3, tfidf__ngram_range=(1, 2), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=  29.9s
[CV] END classifier__colsample_bytree=0.5, classifier__gamma=0, classifier__learning_rate=0.1, clas

/home/akshat/Desktop/main files (i think)/ml course work/ml project/.venv/lib64/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END classifier__colsample_bytree=0.5, classifier__gamma=0, classifier__learning_rate=0.1, classifier__max_depth=3, classifier__min_child_weight=5, classifier__n_estimators=300, classifier__reg_alpha=1, classifier__reg_lambda=1, classifier__subsample=0.85, tfidf__max_df=1.0, tfidf__max_features=60000, tfidf__min_df=3, tfidf__ngram_range=(1, 2), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=  29.9s
[CV] END classifier__colsample_bytree=0.5, classifier__gamma=0, classifier__learning_rate=0.1, classifier__max_depth=3, classifier__min_child_weight=5, classifier__n_estimators=300, classifier__reg_alpha=1, classifier__reg_lambda=1, classifier__subsample=0.85, tfidf__max_df=1.0, tfidf__max_features=60000, tfidf__min_df=3, tfidf__ngram_range=(1, 2), tfidf__stop_words=None, tfidf__sublinear_tf=True; total time=  29.7s
[CV] END classifier__colsample_bytree=0.7, classifier__gamma=0.1, classifier__learning_rate=0.03, classifier__max_depth=6, classifier__min_child_weight=3, class

## Experiment E — Handcrafted review-style features

The handcrafted features below are computed independently for each review. They describe simple writing characteristics; they are not assumed to be reliable indicators of authenticity. Their value must be established through cross-validation.

We compare:
1. numerical style features only;
2. TF-IDF only;
3. TF-IDF + style features.

The feature extraction function is part of the pipeline so the same transformation is applied consistently during cross-validation.

In [11]:
def extract_style_features(texts):
    rows = []
    for text in texts:
        text = str(text)
        words = re.findall(r"\b\w+\b", text)
        word_lengths = [len(w) for w in words]
        rows.append([
            len(text),                                      # character count
            len(words),                                     # word count
            np.mean(word_lengths) if word_lengths else 0,   # average word length
            sum(text.count(p) for p in ".!?"),              # approximate sentence count
            text.count("!"),
            text.count("?"),
            (sum(ch.isupper() for ch in text) /
             sum(ch.isalpha() for ch in text))
                if any(ch.isalpha() for ch in text) else 0,
            (len(set(w.lower() for w in words)) / len(words))
                if words else 0
        ])
    return np.asarray(rows, dtype=float)

style_feature_names = [
    "char_count", "word_count", "avg_word_length",
    "sentence_punctuation_count", "exclamation_count",
    "question_count", "uppercase_ratio", "unique_word_ratio"
]

style_transformer = FunctionTransformer(
    extract_style_features, validate=False, feature_names_out="one-to-one"
)

print("Feature names:", style_feature_names)
print("Example feature matrix shape:", extract_style_features(X_train.iloc[:5]).shape)

Feature names: ['char_count', 'word_count', 'avg_word_length', 'sentence_punctuation_count', 'exclamation_count', 'question_count', 'uppercase_ratio', 'unique_word_ratio']
Example feature matrix shape: (5, 8)


In [12]:
# Use the stop-word setting selected in Experiment A.
# Keep the TF-IDF vocabulary moderate for this first combined-feature test.
feature_experiment_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    max_features=40000,
    stop_words=SELECTED_STOP_WORDS,
    sublinear_tf=True
)

style_only_features = Pipeline([
    ("extract", FunctionTransformer(extract_style_features, validate=False)),
    ("scale", StandardScaler())
])

combined_features = FeatureUnion([
    ("tfidf", feature_experiment_tfidf),
    ("style", style_only_features)
])

# Compare Random Forest and XGBoost on three representations.
feature_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=30, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        min_child_weight=3, subsample=0.85, colsample_bytree=0.7,
        reg_lambda=5, reg_alpha=0.1, objective="binary:logistic",
        eval_metric="logloss", tree_method="hist",
        random_state=RANDOM_STATE, n_jobs=1
    )
}

feature_experiment_rows = []

for model_name, classifier in feature_models.items():
    representations = {
        "TF-IDF only": Pipeline([
            ("tfidf", clone(feature_experiment_tfidf)),
            ("classifier", clone(classifier))
        ]),
        "Style features only": Pipeline([
            ("style", clone(style_only_features)),
            ("classifier", clone(classifier))
        ]),
        "TF-IDF + style features": Pipeline([
            ("features", clone(combined_features)),
            ("classifier", clone(classifier))
        ])
    }

    for representation_name, pipeline in representations.items():
        scores = cross_validate(
            pipeline, X_train, y_train,
            cv=cv, scoring=scoring,
            return_train_score=True,
            n_jobs=N_JOBS, pre_dispatch=N_JOBS
        )
        feature_experiment_rows.append({
            "Model": model_name,
            "Representation": representation_name,
            "Training Macro F1": scores["train_macro_f1"].mean(),
            "CV Macro F1": scores["test_macro_f1"].mean(),
            "Generalization Gap": (
                scores["train_macro_f1"].mean()
                - scores["test_macro_f1"].mean()
            ),
            "CV Macro F1 Std": scores["test_macro_f1"].std(),
            "CV Accuracy": scores["test_accuracy"].mean(),
            "CV Balanced Accuracy": scores["test_balanced_accuracy"].mean(),
            "CV ROC-AUC": scores["test_roc_auc"].mean()
        })

feature_experiment_results = pd.DataFrame(feature_experiment_rows)
display(feature_experiment_results.sort_values(
    "CV Macro F1", ascending=False
).round(4))
feature_experiment_results.to_csv(
    OUTPUT_DIR / "E_handcrafted_feature_comparison.csv", index=False
)

,Model,Representation,Training Macro F1,CV Macro F1,Generalization Gap,CV Macro F1 Std,CV Accuracy,CV Balanced Accuracy,CV ROC-AUC
2,Random Forest,TF-IDF + style features,0.9407,0.8993,0.0414,0.0033,0.8994,0.8994,0.9694
5,XGBoost,TF-IDF + style features,0.9110,0.8982,0.0128,0.0060,0.8982,0.8982,0.9661
0,Random Forest,TF-IDF only,0.9205,0.8770,0.0434,0.0037,0.8773,0.8773,0.9533
3,XGBoost,TF-IDF only,0.8678,0.8526,0.0153,0.0041,0.8527,0.8527,0.9347
4,XGBoost,Style features only,0.8461,0.8397,0.0063,0.0022,0.8397,0.8397,0.9243
1,Random Forest,Style features only,0.9606,0.8377,0.1230,0.0024,0.8377,0.8377,0.9208


## Experiment summary

This table compares the principal searches. The best configurations are selected by CV Macro F1. A smaller generalisation gap is useful context, but is not a substitute for strong validation performance.

The handcrafted-feature experiment is a fixed-configuration comparison and is shown separately because its model settings are not selected through the same randomized search budget.

In [13]:
search_summary = []

search_objects = [
    ("Logistic Regression L1/L2/Elastic Net", logistic_search),
    ("LinearSVC", svm_search),
    ("Random Forest", rf_search),
    ("XGBoost", xgb_search)
]

for name, search in search_objects:
    idx = search.best_index_
    train_f1 = search.cv_results_["mean_train_macro_f1"][idx]
    cv_f1 = search.best_score_
    search_summary.append({
        "Model": name,
        "Training Macro F1": train_f1,
        "CV Macro F1": cv_f1,
        "Generalization Gap": train_f1 - cv_f1,
        "CV Macro F1 Std": search.cv_results_["std_test_macro_f1"][idx],
        "Best Parameters": search.best_params_
    })

search_summary_df = pd.DataFrame(search_summary).sort_values(
    "CV Macro F1", ascending=False
)
display(search_summary_df.drop(columns="Best Parameters").round(4))
search_summary_df.to_json(
    OUTPUT_DIR / "model_improvement_summary.json",
    orient="records", indent=2
)

# Save the best tuned search candidate under a stable filename for downstream notebooks.
best_name, best_search = max(
    search_objects,
    key=lambda item: item[1].best_score_
)
selected_pipeline_path = MODEL_DIR / "selected_pipeline.joblib"
joblib.dump(best_search.best_estimator_, selected_pipeline_path)

selected_model_summary = {
    "selected_model": best_name,
    "best_cv_macro_f1": float(best_search.best_score_),
    "training_macro_f1": float(
        best_search.cv_results_["mean_train_macro_f1"][best_search.best_index_]
    ),
    "generalization_gap": float(
        best_search.cv_results_["mean_train_macro_f1"][best_search.best_index_]
        - best_search.best_score_
    ),
    "best_parameters": {
        key: (value.item() if isinstance(value, np.generic) else value)
        for key, value in best_search.best_params_.items()
    },
    "pipeline_path": str(selected_pipeline_path)
}
with open(OUTPUT_DIR / "selected_model.json", "w") as f:
    json.dump(selected_model_summary, f, indent=2)

print("Selected tuned model:", best_name)
print("Saved generic pipeline:", selected_pipeline_path)


,Model,Training Macro F1,CV Macro F1,Generalization Gap,CV Macro F1 Std
1,LinearSVC,0.9965,0.9438,0.0527,0.0016
0,Logistic Regression L1/L2/Elastic Net,0.9794,0.9398,0.0396,0.0023
2,Random Forest,1.0000,0.9010,0.0990,0.0030
3,XGBoost,0.9140,0.8863,0.0277,0.0030


Selected tuned model: LinearSVC
Saved generic pipeline: ../models/improvement_no_stopwords/selected_pipeline.joblib


## Selection and next steps

1. Review the stop-word comparison before fixing `SELECTED_STOP_WORDS`.
2. Compare the new linear searches against the earlier tuning results.
3. Compare the tree searches against the earlier tree results.
4. Treat the handcrafted-feature comparison as exploratory until its settings are tuned fairly.
5. Select a small set of finalists using CV Macro F1, stability and practical runtime.
6. Only then evaluate the finalists on the held-out test set.

**Important limitation:** the test set has been inspected in earlier project work. Do not describe it as completely untouched or use it repeatedly to choose models. Report this limitation honestly.

## Final operational selection (read after the experiments)

The highest-CV-Macro-F1 tuned candidate in this notebook is LinearSVC (CV Macro F1 approximately 94.38%, with a training–validation gap around 5.27 percentage points). Logistic Regression with L2 regularisation achieved CV Macro F1 approximately 93.98%, with a smaller gap around 3.96 percentage points.

The project retains Logistic Regression as the final website model. The small CV difference is weighed against the smaller generalisation gap and native probability output. LinearSVC remains an important benchmark and compact alternative; its vocabulary-size trade-off is studied in Notebook 09. This is a practical project decision, not proof that Logistic Regression is universally superior.